[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/name-generation-rnn.ipynb)

# Character-Level Name Generation with RNN

This notebook trains a character-level RNN to generate names, then analyzes how many generated names are novel versus memorized from the training data.

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import numpy as np
import wandb
from pytorch_lightning.loggers import WandbLogger

# Import shared utilities from local package
from aiml_notebooks import CharacterTokenizer, NamesDataset, collate_fn, create_dataset, create_dataloaders

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. Th

PyTorch: 2.9.0+cu128
Lightning: 2.5.5


In [2]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'dataset_id': 'palindromes',           # Dataset to use: 'names', 'words', or 'palindromes'
    'seed': 42,                      # Random seed for reproducibility
    'train_split': 0.9,              # Fraction of data for training (rest is validation)
    'batch_size': 128,               # Number of examples per training batch
    
    # Model
    'embedding_dim': 64,             # Size of character embedding vectors
    'hidden_size': 256,              # Number of units in RNN hidden layers
    'num_layers': 2,                 # Number of stacked RNN layers
    'dropout': 0.2,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 30,                # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Generation
    'max_length': 15,                # Maximum characters in generated text
    'temperature': 0.8,              # Sampling randomness (lower=conservative, higher=creative)
    'sample_size': 20,               # Number of examples to generate
    'novelty_sample_size': 100,      # Number of samples for novelty analysis
    
    # Weights & Biases
    'wandb_project': 'name-generation-rnn',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Seed set to 42


42

Next, we'll define all hyperparameters in a CONFIG dictionary and set random seeds for reproducibility.

In [3]:
# Papermill parameters (injected by sweep runner)
# These values will override CONFIG when running sweeps
dataset_id = CONFIG['dataset_id']
embedding_dim = CONFIG['embedding_dim']
hidden_size = CONFIG['hidden_size']
num_layers = CONFIG['num_layers']
dropout = CONFIG['dropout']
learning_rate = CONFIG['learning_rate']
batch_size = CONFIG['batch_size']
max_epochs = CONFIG['max_epochs']
temperature = CONFIG['temperature']
wandb_project = CONFIG['wandb_project']
wandb_run_name = CONFIG['wandb_run_name']

# Update CONFIG with papermill parameters
CONFIG.update({
    'dataset_id': dataset_id,
    'embedding_dim': embedding_dim,
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'max_epochs': max_epochs,
    'temperature': temperature,
    'wandb_project': wandb_project,
    'wandb_run_name': wandb_run_name,
})

print(f"Running with config: {CONFIG}")

Running with config: {'dataset_id': 'palindromes', 'seed': 42, 'train_split': 0.9, 'batch_size': 128, 'embedding_dim': 64, 'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.001, 'max_epochs': 30, 'log_every_n_steps': 20, 'max_length': 15, 'temperature': 0.8, 'sample_size': 20, 'novelty_sample_size': 100, 'wandb_project': 'name-generation-rnn', 'wandb_run_name': None}


These papermill parameters allow hyperparameter sweeps to override the default CONFIG values.

In [4]:
# Create dataset using the factory (handles data loading, tokenization, and splitting)
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id=CONFIG['dataset_id'],
    splits=[CONFIG['train_split'], 1 - CONFIG['train_split']]
)

# Extract the tokenizer from the full dataset for later use
tokenizer = full_dataset.tokenizer

print(f"Dataset: {CONFIG['dataset_id']}")
print(f"Total samples: {len(full_dataset)}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Tokenizer: {tokenizer}")

Dataset: palindromes
Total samples: 10000
Train: 9000, Val: 1000
Tokenizer: CharacterTokenizer(vocab_size=16, chars=.abcdefghijklmno)


Now we'll use the dataset factory to download the names, create the tokenizer, and split the data.

In [ ]:
# Create data loaders using the factory
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['batch_size']
)

# Print sample from the dataset to verify data is accurate
print("\nDataset Sample (first 10 examples):")
print("="*50)
sample_texts = full_dataset.get_texts()[:10]
for i, text in enumerate(sample_texts, 1):
    print(f"{i:2}. {text}")
print("="*50)

Now we'll use the dataloader factory to create batched, shuffled loaders for training and validation.

In [6]:
# Define the character-level RNN model with embedding, RNN layers, and generation method
class NameGeneratorRNN(L.LightningModule):
    def __init__(self, vocab_size: int, embedding_dim: int = 64, hidden_size: int = 256,
                 num_layers: int = 2, dropout: float = 0.2, learning_rate: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        rnn_out, hidden = self.rnn(embedded, hidden)
        rnn_out = self.dropout(rnn_out)
        logits = self.fc(rnn_out)
        return logits, hidden
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self(x)
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))
        self.log('val_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    @torch.no_grad()
    def generate(self, tokenizer, max_length=20, temperature=1.0, num_samples=1):
        """Generate names using the tokenizer."""
        self.eval()
        generated_names = []
        
        for _ in range(num_samples):
            current_idx = tokenizer.get_special_token_idx()
            name_chars = []
            hidden = None
            
            for _ in range(max_length):
                x = torch.tensor([[current_idx]], dtype=torch.long, device=self.device)
                logits, hidden = self(x, hidden)
                probs = F.softmax(logits[0, -1] / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
                next_char = tokenizer.decode_char(next_idx)
                
                if tokenizer.is_special_token(next_char):
                    break
                
                name_chars.append(next_char)
                current_idx = next_idx
            
            generated_names.append(''.join(name_chars))
        
        return generated_names

# Initialize the model with CONFIG hyperparameters
model = NameGeneratorRNN(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
)

Now we'll define our RNN model with embedding, recurrent, and output layers, plus a generation method.

In [7]:
# Initialize Weights & Biases logger
# Check if we're running inside an existing W&B run (e.g., from a sweep)
if wandb.run is not None:
    print(f"Using existing W&B run: {wandb.run.name}")
    wandb_logger = None  # Use existing run instead of creating new logger
else:
    # Create new W&B run
    wandb_logger = WandbLogger(
        project=CONFIG['wandb_project'],
        name=CONFIG['wandb_run_name'],
        config=CONFIG,
    )
    print(f"Created W&B run: {wandb_logger.experiment.name}")

# Train the model using PyTorch Lightning trainer
trainer = L.Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='auto',
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=CONFIG['log_every_n_steps'],
    logger=wandb_logger,
)
trainer.fit(model, train_loader, val_loader)

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()

Created W&B run: hopeful-wave-45


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 1.0 K  | train
1 | rnn       | RNN              | 214 K  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 4.1 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
219 K     Trainable params
0         Non-trainable params
219 K     Total params
0.877     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [8]:
# Generate sample names
generated = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['sample_size']
)
generated = [name.capitalize() for name in generated]
print(", ".join(generated))

# Analyze novelty (new vs existing names)
print("\n" + "="*50)
print("NOVELTY ANALYSIS")
print("="*50)

sample = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['novelty_sample_size']
)
sample = [name for name in sample if name]

# Get original texts from the full dataset for comparison
original_texts_set = set(full_dataset.get_texts())
new_names = [name for name in sample if name not in original_texts_set]
existing_names = [name for name in sample if name in original_texts_set]

# Calculate metrics
novelty_pct = len(new_names) / len(sample) * 100
uniqueness_pct = len(set(sample)) / len(sample) * 100

print(f"Total: {len(sample)}, Unique: {len(set(sample))}")
print(f"✨ NEW: {len(new_names)} ({novelty_pct:.1f}%)")
print(f"♻️  EXISTING: {len(existing_names)} ({len(existing_names)/len(sample)*100:.1f}%)")

print(f"\nNEW: {', '.join([n.capitalize() for n in new_names[:10]])}")
print(f"EXISTING: {', '.join([n.capitalize() for n in existing_names[:10]])}")

# Log to W&B
wandb.log({
    'novelty_percentage': novelty_pct,
    'uniqueness_percentage': uniqueness_pct,
    'total_generated': len(sample),
    'new_names_count': len(new_names),
    'memorized_names_count': len(existing_names),
})

# Log sample names as a table
wandb.log({
    'sample_new_names': wandb.Table(
        columns=['name'],
        data=[[n.capitalize()] for n in new_names[:20]]
    )
})

Gkoddokog, Agklklkga, Jeibaeaib, Limalml, Cjddekkbedjdc, Ecmbbmcbcme, Eamemenmemecmae, Jkcbcmkjkc, Llhklol, Cdhfddfhdcf, Cejmmfemjemcj, Bfmdbodbombfd, Affmeeaaffm, Eajojkeja, Ljkbohhkblob, Mkafflakfm, Mhljldfdllhmjl, Hojnbjbjboo, Bcalacl, Cnecbfbffcbenc

NOVELTY ANALYSIS
Total: 100, Unique: 100
✨ NEW: 100 (100.0%)
♻️  EXISTING: 0 (0.0%)

NEW: Fkljlljjklf, Gfeeffe, Gkiceeekcig, Ahggciggcag, Kghmombmoohgk, Dlckdlc, Jlmjjjmjl, Ggjfjfdjfgg, Kmbfkmokfmbkb, Iflgicfilcgfii
EXISTING: 


Finally, we'll generate sample names and analyze how many are novel versus memorized from training.

---

## Bonus: Testing with Palindromes (Long-Range Dependencies)

The palindrome dataset is specifically designed to expose RNN limitations. To use it, change the config above to `'dataset_id': 'palindromes'` and re-run the notebook.

**Why palindromes are hard for RNNs:**
- Palindromes require remembering the start when generating the end (e.g., "abcdcba")
- This creates dependencies spanning 10+ steps, which RNNs struggle with due to vanishing gradients
- You'll likely see the RNN generate sequences that *look* plausible but fail the palindrome constraint

**What you'll observe:**
- RNN will learn the character distribution but fail to maintain long-range structure
- Generated sequences will be gibberish, not valid palindromes
- Later architectures (LSTMs, Transformers) handle this much better

Try it! Change dataset_id to 'palindromes' and watch the RNN struggle.